# PFF → Zarr v3: Conversion, Exploration, and Analysis

This notebook demonstrates the end-to-end workflow for converting PanoSETI PFF data to
Zarr v3 stores using `pypff.zarr`, then working with the data via xarray and dask.

## What you'll learn
1. Opening a PFF run with `PanosetiRun` and inspecting its products
2. Converting PFF → Zarr in one call (`convert_run`)
3. Working with the resulting xarray Dataset — dimensions, variables, timestamps
4. Visualizing images and time series
5. Baseline / pedestal subtraction (pulse-height data)
6. Block-median subtraction (image data)
7. Using header arrays for diagnostics

## Prerequisites
```bash
uv sync  # in the pypff repo, installs pypff[zarr]
```
or from the pipeline repo:
```bash
uv sync  # in panoseti_zarr_seqera/
```

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
import zarr

from pypff.io2 import PanosetiRun, PFFSequence
from pypff.zarr import convert_run

## 1. Point to your observation directory

A `.pffd` directory holds all `.pff` files for one observing run.
Set `OBS_DIR` to your run directory and `OUT_DIR` to where the Zarr stores will be written.

In [ ]:
# ── configure paths here ─────────────────────────────────────────────────────
OBS_DIR = Path("../../panoseti_zarr_seqera/obs_TEST.pffd")  # adjust to your run
OUT_DIR = Path("/tmp/L0_zarr_demo")
# ─────────────────────────────────────────────────────────────────────────────

assert OBS_DIR.exists(), f"Run directory not found: {OBS_DIR}"

## 2. Inspect the run before converting

`PanosetiRun` scans the `.pffd` directory and groups `.pff` files by
`(data_product, module)` pair.  Each group becomes one `PFFSequence`.

In [ ]:
run = PanosetiRun(OBS_DIR)
products = run.list_products()
print(f"Found {len(products)} data product(s) in {OBS_DIR.name}:")

for name in products:
    seq: PFFSequence = run.get_product(name)
    conf = seq.frame_config
    print(f"\n  {name}")
    print(f"    files  : {len(seq.file_paths)}")
    print(f"    frames : {len(seq):,}")
    print(f"    shape  : {conf.image_shape}  dtype={conf.dtype_str}")
    print(f"    header : {'module-level (quabo_0..3)' if conf.format_name != 'ph256_one_level' else 'single-level'}")

### 2a. Peek at raw timestamps (int64 nanoseconds)

All timestamps in `pypff` are `int64` nanoseconds since the Unix epoch — never
Python floats.  `float64` only has ~15–16 significant digits; a Unix timestamp
in nanoseconds is ~19 digits, so dividing first loses nanosecond precision.

**Rule**: subtract a reference epoch in integer space, *then* divide.

In [ ]:
seq = run.get_product(products[0])
ts_ns = seq.timestamps()          # int64 array, ns since epoch
ts_dt = seq.timestamps(as_datetime=True)  # datetime64[ns] zero-copy view

print(f"First timestamp  : {ts_dt[0]}")
print(f"Last  timestamp  : {ts_dt[-1]}")
t0_ns = int(ts_ns[0])
dt_s = (ts_ns - t0_ns) / 1e9     # relative seconds — no precision loss
print(f"Duration         : {dt_s[-1]:.3f} s")
print(f"Mean frame rate  : {len(ts_ns) / dt_s[-1]:.1f} Hz")

## 3. Convert PFF → Zarr v3

`convert_run` streams every product from the run into one `.zarr` store per
`(data_product, module)` pair.  It returns the list of created store paths.

Zarr v3 store layout:
```
root/
  images/          (T, H, W)  — main pixel data, chunked along time
  unix_t_ns/       (T,)  int64  — nanosecond timestamps
  pkt_num/         (T,)  uint32  ─┐
  pkt_tai/         (T,)  uint16   │ per-frame header fields
  pkt_nsec/        (T,)  uint32   │ (flat at root → visible in xarray)
  tv_sec/          (T,)  int64    │
  tv_usec/         (T,)  uint32  ─┘
  quabo_num/       (T,)  uint8    — ph256 single-level only
  quabo_0_pkt_num/ (T,)  uint32  ─┐ img/ph1024 module-level
  quabo_0_pkt_tai/ (T,)  uint16   │ (four quabo sub-sets)
  ...                             ─┘
```

In [ ]:
import time

t0 = time.monotonic()
stores = convert_run(
    run,
    OUT_DIR,
    codec="zstd",    # zstd (default), blosc-lz4, gzip, or none
    level=3,         # compression level
    # time_chunk=4096  # override auto chunk size
)
elapsed = time.monotonic() - t0

total_pff  = sum(f.stat().st_size for p in products for f in run.get_product(p).file_paths)
total_zarr = sum(f.stat().st_size for s in stores for f in s.rglob("*") if f.is_file())

print(f"Converted {len(stores)} store(s) in {elapsed:.1f} s")
print(f"  PFF  : {total_pff  / 1024**2:.1f} MB")
print(f"  Zarr : {total_zarr / 1024**2:.1f} MB   ({total_pff / total_zarr:.2f}× compression ratio)")
for s in stores:
    print(f"  → {s}")

## 4. Open a store with xarray

`xr.open_zarr` reads all root-level arrays as dataset variables.  The `time`
dimension is shared across images, timestamps, and all header fields — enabling
label-aligned selection and computation.

In [ ]:
# Open the first store (or choose a specific one by data product name)
store_path = stores[0]
ds = xr.open_zarr(str(store_path))
print(ds)

In [ ]:
# Attach a human-readable time coordinate for plotting
# unix_t_ns is int64 ns; view it as datetime64 (zero-copy)
ds = ds.assign_coords(
    time_utc=("time", ds["unix_t_ns"].values.view("datetime64[ns]"))
)
print("Coordinates:", dict(ds.coords))
print("Data variables:", list(ds.data_vars))

## 5. Visualize images

Each frame is a 2-D pixel array.  `ds.images` has shape `(T, H, W)` with
dask arrays under the hood — `.values` triggers the actual compute.

In [ ]:
n_show = min(9, len(ds.time))
step   = max(1, len(ds.time) // n_show)
frames = ds.images.isel(time=slice(0, n_show * step, step)).values  # (n_show, H, W)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
vmin, vmax = np.percentile(frames, [1, 99])

for ax, frame in zip(axes.flat, frames):
    ax.imshow(frame, origin="upper", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.axis("off")

fig.suptitle(f"{store_path.name}\nEvery {step}th frame", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Time series — mean photon count per frame

In [ ]:
mean_count = ds.images.mean(dim=["y", "x"]).compute()
t_rel_s = (ds["unix_t_ns"].values - int(ds["unix_t_ns"].values[0])) / 1e9

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_rel_s, mean_count.values, lw=0.5)
ax.set_xlabel("Time since run start (s)")
ax.set_ylabel("Mean pixel value")
ax.set_title(f"Mean pixel value over time — {ds.attrs['data_product']} module {ds.attrs['module']}")
plt.tight_layout()
plt.show()

## 7. Working with header arrays

Header fields (`pkt_num`, `pkt_nsec`, `tv_sec`, etc.) are stored as 1-D arrays
with the same `time` dimension as the images.  This lets you filter frames by
hardware packet number, check for missing packets, or correlate headers with pixel data.

In [ ]:
# Header dtypes are shrunk to save space (see pypff.zarr._HEADER_DTYPES)
for var in ds.data_vars:
    if var not in ("images", "unix_t_ns"):
        print(f"  {var:25s} dtype={ds[var].dtype}")

In [ ]:
# Check for missing packets: pkt_num should be monotonically increasing
pkt_num = ds["pkt_num"].values.astype(np.int64)
gaps = np.diff(pkt_num)
missing = np.where(gaps > 1)[0]

print(f"Total frames   : {len(pkt_num):,}")
print(f"Expected gaps  : {(gaps == 1).sum():,}")
print(f"Missing packets: {missing.size}")
if missing.size:
    print(f"  First gap at frame {missing[0]}, skipped {gaps[missing[0]] - 1} packets")

## 8. Baseline / pedestal subtraction (pulse-height data)

The pedestal is the DC offset from each pixel's pre-amplifier.  For `ph*`
products we:
1. Add a fixed baseline offset (800 ADC counts) so subtraction doesn't underflow
2. Compute the per-pixel median across all time frames
3. Subtract the median pedestal
4. Apply a sigma threshold to isolate real pulses

In [ ]:
# Skip this cell if the store is an image product
if "ph" not in str(ds.attrs.get("data_product", "")):
    print("Not a pulse-height product — skipping this cell.")
else:
    BASELINE_OFFSET  = 800
    SIGMA_THRESHOLD  = 5

    imgs = ds.images.astype("int32") + BASELINE_OFFSET  # avoid int16 underflow

    # For ph1024 (32×32): swap quabo quadrant positions
    if imgs.sizes["y"] == 32:
        imgs_np = imgs.values.copy()
        imgs_np[:, :16, 16:], imgs_np[:, 16:, :16] = (
            imgs_np[:, 16:, :16].copy(),
            imgs_np[:, :16, 16:].copy(),
        )
        import dask.array as da
        imgs = xr.DataArray(
            da.from_array(imgs_np, chunks=imgs.data.chunksize),
            dims=imgs.dims, coords=imgs.coords,
        )

    pedestal  = imgs.median(dim="time")
    ped_sub   = imgs - pedestal
    sigma     = ped_sub.std("time")
    threshold = sigma * SIGMA_THRESHOLD
    above     = ped_sub.where(ped_sub > threshold)

    # Trigger rate per pixel (fraction of frames above threshold)
    trigger_rate = (ped_sub > threshold).mean("time").compute()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    im0 = axes[0].imshow(pedestal.compute().values, cmap="coolwarm")
    axes[0].set_title("Pedestal (median across time)")
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(sigma.compute().values, cmap="plasma")
    axes[1].set_title("RMS noise (σ)")
    plt.colorbar(im1, ax=axes[1])

    im2 = axes[2].imshow(trigger_rate.values * 100, cmap="hot")
    axes[2].set_title(f"{SIGMA_THRESHOLD}σ trigger rate (%)")
    plt.colorbar(im2, ax=axes[2])

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 9. Block-median subtraction (image data)

For `img*` products, each quabo's 16×16 region has non-uniform gain.  The
8×8 block-median approach estimates and removes the spatial background:
1. Convert to `int32` (avoid uint16 overflow on subtraction)
2. Compute the median of each 8×8 block over a strided temporal subset
3. Tile back to full frame size and subtract
4. Apply a temporal median to remove detector drifts

In [ ]:
if "img" not in str(ds.attrs.get("data_product", "")):
    print("Not an image product — skipping this cell.")
else:
    FRAME_STEP  = 200   # use every 200th frame to estimate background
    BLOCK_SIZE  = 8     # pixels per block side
    ADC_TO_PE   = 1.5   # ADC counts per photo-electron

    imgs = ds.images.astype("int32")  # (T, H, W)
    H, W = imgs.sizes["y"], imgs.sizes["x"]
    nb = H // BLOCK_SIZE            # number of blocks per axis (typically 4 for 32×32)

    # Sub-sample frames for background estimation
    stride_imgs = imgs[::FRAME_STEP].values  # (T', H, W) — triggers compute

    # Reshape to (T', nb, BS, nb, BS) then median over blocks
    BS = BLOCK_SIZE
    blocks = stride_imgs.reshape(-1, nb, BS, nb, BS)
    block_medians = np.median(blocks, axis=(2, 4))  # (T', nb, nb)
    temporal_super = np.median(block_medians, axis=0)  # (nb, nb)

    # Upsample back to full frame size via repeat
    background = np.repeat(np.repeat(temporal_super, BS, axis=0), BS, axis=1)  # (H, W)

    # Subtract and convert to photo-electrons
    imgs_pe = (imgs.values - background[np.newaxis]) / ADC_TO_PE

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    im0 = axes[0].imshow(background, cmap="coolwarm")
    axes[0].set_title("Estimated spatial background (ADC)")
    plt.colorbar(im0, ax=axes[0])

    mean_frame_raw = imgs.mean("time").values
    im1 = axes[1].imshow(mean_frame_raw, cmap="gray")
    axes[1].set_title("Mean frame (raw, ADC)")
    plt.colorbar(im1, ax=axes[1])

    mean_frame_sub = imgs_pe.mean(axis=0)
    im2 = axes[2].imshow(mean_frame_sub, cmap="gray")
    axes[2].set_title("Mean frame (background-subtracted, p.e.)")
    plt.colorbar(im2, ax=axes[2])

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 10. Slicing and selection

Because every variable shares the `time` dimension, you can select time ranges
and get aligned images + headers + timestamps in one operation.

In [ ]:
# Select the first 100 frames
sub = ds.isel(time=slice(0, 100))

# Timestamps of those frames
t_sub_ns = sub["unix_t_ns"].values
t_sub_dt = t_sub_ns.view("datetime64[ns]")

print(f"Sub-selection: {len(sub.time)} frames")
print(f"  Start : {t_sub_dt[0]}")
print(f"  End   : {t_sub_dt[-1]}")
if "pkt_num" in sub:
    print(f"  pkt_num range: {int(sub.pkt_num.values[0])} – {int(sub.pkt_num.values[-1])}")

In [ ]:
# Pixel-level time series for the brightest pixel in the mean frame
mean_frame = ds.images.mean("time").compute().values
yi, xi = np.unravel_index(np.argmax(mean_frame), mean_frame.shape)

pixel_ts = ds.images.isel(y=yi, x=xi).compute().values.astype(float)
t_rel_s  = (ds["unix_t_ns"].values - int(ds["unix_t_ns"].values[0])) / 1e9

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_rel_s, pixel_ts, lw=0.5)
ax.set_xlabel("Time since run start (s)")
ax.set_ylabel("Pixel value (ADC)")
ax.set_title(f"Brightest pixel time series — y={yi}, x={xi}")
plt.tight_layout()
plt.show()

## 11. Direct zarr access (without xarray)

If you prefer working with raw numpy arrays, open the zarr group directly.
This is useful for ad-hoc slicing of very large stores without loading everything.

In [ ]:
zg = zarr.open_group(str(store_path), mode="r")

print("Root arrays:")
for key in sorted(zg.array_keys()):
    arr = zg[key]
    print(f"  {key:30s} {str(arr.shape):20s} {arr.dtype}")

# Read 10 frames starting at frame 50
imgs_raw = zg["images"][50:60]          # shape (10, H, W)
ts_raw   = zg["unix_t_ns"][50:60]       # int64 ns
print(f"\nLoaded slice: images={imgs_raw.shape} ts={ts_raw.shape}")

## 12. Re-opening with dask for out-of-core processing

For very large stores (tens of millions of frames), use dask to process
without loading everything into RAM. `xr.open_zarr` already returns dask-backed
arrays; call `.compute()` only when you need the actual values.

In [ ]:
# All operations below are lazy — no data is read until .compute()
ds_lazy = xr.open_zarr(str(store_path))  # dask arrays by default

# Lazy mean across time — reads each chunk once
mean_lazy = ds_lazy.images.mean("time")
print("mean_lazy (still lazy):", mean_lazy)

# Trigger compute — reads compressed chunks, decompresses, averages
mean_vals = mean_lazy.compute()
print(f"mean computed: {mean_vals.values.shape}  max={float(mean_vals.max().values):.1f}")

In [ ]:
# Optional: connect to a Dask distributed cluster for multi-core compute
# Uncomment and set the scheduler address from your Nextflow run

# from dask.distributed import Client
# client = Client("tcp://headnode:8786")  # replace with your scheduler address
# print(client)

# # Now .compute() runs on the distributed cluster
# mean_dist = ds_lazy.images.mean("time").compute()

# client.close()

## 13. Store metadata and provenance

Every Zarr store carries root-level attributes that record the source PFF files,
frame configuration, and format version.

In [ ]:
import json

zg = zarr.open_group(str(store_path), mode="r")
attrs = dict(zg.attrs)

print(f"panoseti_pff_zarr_version : {attrs['panoseti_pff_zarr_version']}")
print(f"data_product              : {attrs['data_product']}")
print(f"module                    : {attrs['module']}")
print(f"total_frames              : {attrs['total_frames']:,}")
print(f"header_format             : {attrs['header_format']}")
print(f"source files              : {len(attrs['source_pff_files'])}")
for f in attrs['source_pff_files']:
    print(f"  {f}")
print(f"\nframe_config:")
print(json.dumps(attrs['frame_config'], indent=2))

## Summary

| Task | One-liner |
|------|----------|
| Open a run | `run = PanosetiRun("obs.pffd")` |
| List products | `run.list_products()` |
| Convert to Zarr | `stores = convert_run(run, "L0_zarr/")` |
| Open with xarray | `ds = xr.open_zarr(str(store))` |
| Add datetime coord | `ds.assign_coords(time_utc=("time", ds.unix_t_ns.values.view("datetime64[ns]")))` |
| Get timestamps | `ds.unix_t_ns.values` (int64 ns) or `.view("datetime64[ns]")` |
| Slice 100 frames | `ds.isel(time=slice(0, 100))` |
| Pedestal | `ds.images.median("time")` |
| Mean frame | `ds.images.mean("time").compute()` |
| Header check | `ds.pkt_num` (uint32), `ds.quabo_num` (uint8, ph256 only) |